In [ ]:
"""
AutoCAD Equipment Counter — exports results to CSV
Counts labels matching: E<n>.<m>  or  ET<n>.<m>  (e.g. E2.1, ET2.3)
Output: equipment_count.csv  (same folder as this script)
"""

import re
import csv
import os
from collections import defaultdict
from datetime import datetime
from pyautocad import Autocad

acad = Autocad(create_if_not_exists=True)
print(f"Connected to AutoCAD: {acad.doc.Name}\n")

PATTERN = re.compile(r'(ET?\d+\.\d+)', re.IGNORECASE)

def clean_mtext(raw):
    text = raw
    text = re.sub(r'\\[a-zA-Z][^;\\{}]*;', '', text)
    text = text.replace('\\P', '\n').replace('\\p', '\n')
    text = re.sub(r'\\[^a-zA-Z]', '', text)
    text = text.replace('{', '').replace('}', '')
    return text.strip()

def extract_labels(text, counts):
    text = clean_mtext(text)
    for match in PATTERN.finditer(text):
        counts[match.group(1).upper()] += 1

counts = defaultdict(int)
visited_blocks = set()

def process_object(obj):
    try:
        obj_type = obj.ObjectName
    except Exception:
        return

    if obj_type in ("AcDbMText", "AcDbText"):
        try:
            extract_labels(obj.TextString, counts)
        except Exception:
            pass

    elif obj_type in ("AcDbBlockReference", "AcDbMInsertBlock"):
        try:
            for att in obj.GetAttributes():
                try:
                    extract_labels(att.TextString, counts)
                except Exception:
                    pass
        except Exception:
            pass

        try:
            block_name = obj.EffectiveName
            if block_name not in visited_blocks:
                visited_blocks.add(block_name)
                for ent in acad.doc.Blocks.Item(block_name):
                    process_object(ent)
        except Exception:
            pass

# ── Walk ModelSpace ───────────────────────────────────────────────────────
for obj in acad.doc.ModelSpace:
    process_object(obj)

# ── Walk all Layouts ──────────────────────────────────────────────────────
try:
    for layout in acad.doc.Layouts:
        try:
            for obj in layout.Block:
                process_object(obj)
        except Exception:
            pass
except Exception:
    pass

# ── Sort ──────────────────────────────────────────────────────────────────
def sort_key(label):
    prefix = "ET" if label.startswith("ET") else "E"
    rest   = label[len(prefix):]
    parts  = rest.split(".")
    return (prefix, int(parts[0]), int(parts[1]) if len(parts) > 1 else 0)

sorted_labels = sorted(counts, key=sort_key)
total = sum(counts.values())

# ── Print to console ──────────────────────────────────────────────────────
if not counts:
    print("No labels found.")
else:
    print(f"{'Label':<12}  {'Count':>5}")
    print("-" * 20)
    for label in sorted_labels:
        print(f"{label:<12}  {counts[label]:>5}")
    print("-" * 20)
    print(f"{'TOTAL':<12}  {total:>5}")

# ── Export to CSV ─────────────────────────────────────────────────────────
drawing_name = os.path.splitext(acad.doc.Name)[0]
timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path     = os.path.join(os.path.dirname(os.path.abspath(__file__)),
                            f"{drawing_name}_equipment_{timestamp}.csv")


with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    # utf-8-sig adds BOM so Excel opens it correctly
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["Drawing", "Label", "Count"])
    for label in sorted_labels:
        writer.writerow([acad.doc.Name, label, counts[label]])
    writer.writerow(["", "TOTAL", total])

print(f"\nCSV exported → {csv_path}")
print("Done.")

Connected to AutoCAD: Drawing2.dwg

Label         Count
--------------------
E1.1             18
E1.2             14
E2.1             24
E2.2             12
E3.1             16
E3.2             14
E4.1              8
E4.2             18
E5.1             12
E5.2             12
E6.1             20
E6.2             12
E7.1             28
E7.2             14
E8.1             14
E8.2             14
E9.1             12
E9.2             14
E10.1            12
E10.2            14
E11.1            10
E11.2            10
E12.1            14
E12.2            16
E13.1            14
E13.2            14
E14.1            14
E14.2            14
E15.1            20
E15.2            22
E16.1            20
E16.2            12
E17.1            20
E17.2            14
E18.1            16
E18.2            14
E19.1            12
E19.2            18
E20.1            18
E20.2            36
E21.1            14
E21.2            18
E22.1            14
E22.2            18
E23.1            14
E23.2            20
E24

NameError: name '__file__' is not defined

In [31]:
import re
import csv
import os
from collections import defaultdict
from datetime import datetime
from pyautocad import Autocad

# Connect to AutoCAD
acad = Autocad(create_if_not_exists=True)
print(f"Connected to AutoCAD: {acad.doc.Name}\n")

# Pattern to match E#.# or ET#.# (case insensitive)
PATTERN = re.compile(r'(ET?\d+\.\d+)', re.IGNORECASE)

def clean_mtext(raw):
    """Clean MTEXT formatting codes"""
    text = raw
    text = re.sub(r'\\[a-zA-Z][^;\\{}]*;', '', text)  # Remove formatting codes
    text = text.replace('\\P', '\n').replace('\\p', '\n')  # Handle line breaks
    text = re.sub(r'\\[^a-zA-Z]', '', text)  # Remove remaining backslash codes
    text = text.replace('{', '').replace('}', '')  # Remove braces
    return text.strip()

def extract_labels(text, counts):
    """Extract and count equipment labels from text"""
    text = clean_mtext(text)
    for match in PATTERN.finditer(text):
        counts[match.group(1).upper()] += 1

# Initialize counters
counts = defaultdict(int)
visited_blocks = set()

def process_object(obj):
    """Process AutoCAD objects to extract text"""
    try:
        obj_type = obj.ObjectName
    except Exception:
        return

    # Process single line text and multiline text
    if obj_type in ("AcDbMText", "AcDbText"):
        try:
            extract_labels(obj.TextString, counts)
        except Exception:
            pass

    # Process blocks and attributes
    elif obj_type in ("AcDbBlockReference", "AcDbMInsertBlock"):
        # Check block attributes
        try:
            for att in obj.GetAttributes():
                try:
                    extract_labels(att.TextString, counts)
                except Exception:
                    pass
        except Exception:
            pass

        # Recursively process nested blocks
        try:
            block_name = obj.EffectiveName
            if block_name not in visited_blocks:
                visited_blocks.add(block_name)
                for ent in acad.doc.Blocks.Item(block_name):
                    process_object(ent)
        except Exception:
            pass

# ── Walk ModelSpace ───────────────────────────────────────────────────────
print("Scanning ModelSpace...")
for obj in acad.doc.ModelSpace:
    process_object(obj)

# ── Walk all Layouts (PaperSpace) ─────────────────────────────────────────
print("Scanning Layouts...")
try:
    for layout in acad.doc.Layouts:
        try:
            for obj in layout.Block:
                process_object(obj)
        except Exception:
            pass
except Exception:
    pass

# ── Sort labels naturally ──────────────────────────────────────────────────
def sort_key(label):
    """Sort key for equipment labels (E vs ET, then numbers)"""
    prefix = "ET" if label.startswith("ET") else "E"
    rest = label[len(prefix):]
    parts = rest.split(".")
    return (prefix, int(parts[0]), int(parts[1]) if len(parts) > 1 else 0)

sorted_labels = sorted(counts, key=sort_key)
total = sum(counts.values())

# ── Print results to console ──────────────────────────────────────────────
print("\n" + "="*30)
print("EQUIPMENT COUNT SUMMARY")
print("="*30)

if not counts:
    print("No equipment labels found.")
else:
    print(f"{'Label':<12}  {'Count':>5}")
    print("-" * 20)
    for label in sorted_labels:
        print(f"{label:<12}  {counts[label]:>5}")
    print("-" * 20)
    print(f"{'TOTAL':<12}  {total:>5}")

# ── Export to CSV in specified folder ─────────────────────────────────────
# Your specified output folder
output_folder = r"C:\Users\AMIDOU MAIGA\OneDrive - Institut 2IE\C_MAIGA Amidou\PyAutoCad"

# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Generate filename with drawing name and timestamp
drawing_name = os.path.splitext(acad.doc.Name)[0]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"{drawing_name}_equipment_{timestamp}.csv"
csv_path = os.path.join(output_folder, csv_filename)

# Write CSV file
with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    
    # Write header
    writer.writerow(["Drawing Name", "Label", "Count"])
    writer.writerow([acad.doc.Name, "Scan Date", timestamp])
    writer.writerow([])  # Empty row for spacing
    
    # Write data
    for label in sorted_labels:
        writer.writerow([acad.doc.Name, label, counts[label]])
    
    # Write total
    writer.writerow([])
    writer.writerow(["", "TOTAL", total])
    writer.writerow(["", "Unique Labels", len(sorted_labels)])

print(f"\n✓ CSV file successfully exported!")
print(f"  Location: {csv_path}")
print(f"  File name: {csv_filename}")
print("\nDone.")

Connected to AutoCAD: Drawing2.dwg

Scanning ModelSpace...
Scanning Layouts...

EQUIPMENT COUNT SUMMARY
Label         Count
--------------------
E1.1             18
E1.2             14
E2.1             24
E2.2             12
E3.1             16
E3.2             14
E4.1              8
E4.2             18
E5.1             12
E5.2             12
E6.1             20
E6.2             12
E7.1             28
E7.2             14
E8.1             14
E8.2             14
E9.1             12
E9.2             14
E10.1            12
E10.2            14
E11.1            10
E11.2            10
E12.1            14
E12.2            16
E13.1            14
E13.2            14
E14.1            14
E14.2            14
E15.1            20
E15.2            22
E16.1            20
E16.2            12
E17.1            20
E17.2            14
E18.1            16
E18.2            14
E19.1            12
E19.2            18
E20.1            18
E20.2            36
E21.1            14
E21.2            18
E22.1          

In [33]:
"""
AutoCAD Equipment Counter — clean CSV export with proper columns
Counts labels matching: E<n>.<m>  or  ET<n>.<m>  (e.g. E2.1, ET2.3)
"""

import re
import csv
import os
from collections import defaultdict
from datetime import datetime
from pyautocad import Autocad

# ── Connect ───────────────────────────────────────────────────────────────
acad = Autocad(create_if_not_exists=True)
print(f"Connected to AutoCAD: {acad.doc.Name}\n")

PATTERN = re.compile(r'(ET?\d+\.\d+)', re.IGNORECASE)

def clean_mtext(raw):
    text = raw
    text = re.sub(r'\\[a-zA-Z][^;\\{}]*;', '', text)
    text = text.replace('\\P', '\n').replace('\\p', '\n')
    text = re.sub(r'\\[^a-zA-Z]', '', text)
    text = text.replace('{', '').replace('}', '')
    return text.strip()

def extract_labels(text, counts):
    text = clean_mtext(text)
    for match in PATTERN.finditer(text):
        counts[match.group(1).upper()] += 1

counts = defaultdict(int)
visited_blocks = set()

def process_object(obj):
    try:
        obj_type = obj.ObjectName
    except Exception:
        return

    if obj_type in ("AcDbMText", "AcDbText"):
        try:
            extract_labels(obj.TextString, counts)
        except Exception:
            pass

    elif obj_type in ("AcDbBlockReference", "AcDbMInsertBlock"):
        try:
            for att in obj.GetAttributes():
                try:
                    extract_labels(att.TextString, counts)
                except Exception:
                    pass
        except Exception:
            pass

        try:
            block_name = obj.EffectiveName
            if block_name not in visited_blocks:
                visited_blocks.add(block_name)
                for ent in acad.doc.Blocks.Item(block_name):
                    process_object(ent)
        except Exception:
            pass

# ── Scan ──────────────────────────────────────────────────────────────────
print("Scanning ModelSpace...")
for obj in acad.doc.ModelSpace:
    process_object(obj)

print("Scanning Layouts...")
try:
    for layout in acad.doc.Layouts:
        try:
            for obj in layout.Block:
                process_object(obj)
        except Exception:
            pass
except Exception:
    pass

# ── Sort ──────────────────────────────────────────────────────────────────
def sort_key(label):
    prefix = "ET" if label.startswith("ET") else "E"
    rest   = label[len(prefix):]
    parts  = rest.split(".")
    return (prefix, int(parts[0]), int(parts[1]) if len(parts) > 1 else 0)

sorted_labels = sorted(counts, key=sort_key)
total         = sum(counts.values())

# ── Console summary ───────────────────────────────────────────────────────
print("\n" + "=" * 30)
print("EQUIPMENT COUNT SUMMARY")
print("=" * 30)

if not counts:
    print("No equipment labels found.")
else:
    print(f"{'Label':<12}  {'Count':>5}")
    print("-" * 20)
    for label in sorted_labels:
        print(f"{label:<12}  {counts[label]:>5}")
    print("-" * 20)
    print(f"{'TOTAL':<12}  {total:>5}")
    print(f"{'Unique labels':<12}  {len(sorted_labels):>5}")

# ── CSV export ────────────────────────────────────────────────────────────
output_folder = r"C:\Users\AMIDOU MAIGA\OneDrive - Institut 2IE\C_MAIGA Amidou\PyAutoCad"
os.makedirs(output_folder, exist_ok=True)

drawing_name  = os.path.splitext(acad.doc.Name)[0]
timestamp     = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename  = f"{drawing_name}_equipment_{timestamp}.csv"
csv_path      = os.path.join(output_folder, csv_filename)

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")

    # ── Metadata block (each field in its own column) ─────────────────
    writer.writerow(["Drawing Name",  "Scan Date",       "Total Labels", "Unique Labels"])
    writer.writerow([acad.doc.Name,   timestamp,          total,          len(sorted_labels)])

    writer.writerow([])  # blank separator row

    # ── Column headers ────────────────────────────────────────────────
    writer.writerow(["No.", "Label", "Count", "% of Total"])

    # ── Data rows ─────────────────────────────────────────────────────
    for i, label in enumerate(sorted_labels, start=1):
        count   = counts[label]
        percent = f"{count / total * 100:.1f}%" if total else "0.0%"
        writer.writerow([i, label, count, percent])

    # ── Summary footer ────────────────────────────────────────────────
    writer.writerow([])
    writer.writerow(["", "TOTAL", total, "100.0%"])

print(f"\n✓ CSV exported successfully!")
print(f"  Location : {csv_path}")
print(f"  File     : {csv_filename}")
print("\nDone.")

Connected to AutoCAD: Drawing3.dwg

Scanning ModelSpace...
Scanning Layouts...

EQUIPMENT COUNT SUMMARY
Label         Count
--------------------
E1.1             18
E1.2             14
E2.1             24
E2.2             12
E3.1             16
E3.2             14
E4.1              8
E4.2             18
E5.1             12
E5.2             12
E6.1             20
E6.2             12
E7.1             28
E7.2             14
E8.1             14
E8.2             14
E9.1             12
E9.2             14
E10.1            12
E10.2            14
E11.1            10
E11.2            10
E12.1            14
E12.2            16
E13.1            14
E13.2            14
E14.1            14
E14.2            14
E15.1            20
E15.2            22
E16.1            20
E16.2            12
E17.1            20
E17.2            14
E18.1            16
E18.2            14
E19.1            12
E19.2            18
E20.1            18
E20.2            36
E21.1            14
E21.2            18
E22.1          